# DSA5208 Assignment 1.3: Network Partition

### Group Members

- Ashley Lee Ruoxin
- Koh Si Xing
- Kevin Royce Thomson

## 1. Project Setup

In [ ]:
# Only run if needed
# %pip install pymongo

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.errors import ExecutionTimeout, PyMongoError, WTimeoutError
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

import subprocess
import time
import pandas as pd

### 1.1 Database Connections

Identical to the other two notebooks, so results are comparable.

- `client`  connects to the replica set dsa5208. Writes always go to the primary.
- `direct` holds one direct connection per node. During a partition these still work, because
  the partition only blocks node-to-node traffic, never the client's path to a node. This is what
  lets us query the minority node's own stale copy of the data.

In [2]:
REPLICA_SET_URI = "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208"
DB_NAME = "university"
NODES = ["mongo1", "mongo2", "mongo3"]

client = MongoClient(REPLICA_SET_URI, serverSelectionTimeoutMS=5000)

direct = {
    node: MongoClient(f"mongodb://{node}:27017/?directConnection=true",
                      serverSelectionTimeoutMS=3000)
    for node in NODES
}

### 1.2 Consistency Configurations

The same two combinations as the failover notebook: the strongest, and the weakest offered by MongoDB.

| Configuration | Write Concern | Read Concern | Meaning |
|---|---|---|---|
| `majority` | `majority` | `majority` | Confirm writes only after a majority of nodes store them; return safely majority-committed data. |
| `w1-local` | `1` | `local` | Confirm writes after only the primary stores them; return the selected replica’s locally available data. |

In summary:

- `w="majority"` provides stronger write durability but may take longer.
- `w=1` confirms writes faster but replicas may not have received them yet.
- `readConcern="majority"` returns a safe majority-committed value, but not necessarily the latest value written by the same client.
- `readConcern="local"` returns the selected node’s current value immediately, which may be stale.

In [3]:
CONFIGS = {
    "majority": {"write_concern": "majority", "read_concern": "majority"},
    "w1-local": {"write_concern": 1, "read_concern": "local"}
}

READ_TIMEOUT_MS = 3000      # how long a read may wait before we record TIMEOUT
WTIMEOUT_MS = 3000          # how long a write waits for confirmation from other nodes
NUM_READS = 10              # reads per loop, so we hit both secondaries
CATCHUP_SECONDS = 2         # safety margin for a node to catch up after the partition heals
TIMEOUT = "TIMEOUT"

MINORITY_NODE = "mongo3"    # the node we isolate, kept the same across all experiments

### 1.3 Helper Functions

#### 1.3.1 Collections and test data

In [4]:
def get_collection(collection_name, config_name, target="primary"):
    """
    Return a collection handle with the config's write and read concern applied.

    target:
        "primary"    - reads come from the primary
        "secondary"  - reads come from any secondary
        a node name  - reads come directly from that node, with local read concern,
                       so we see exactly what that node has stored

    Writes always go to the primary, whatever the target.
    """
    config = CONFIGS[config_name]

    if target in NODES:
        db = direct[target].get_database(DB_NAME, read_concern=ReadConcern("local"))
        return db[collection_name]

    read_preference = {
        "primary": ReadPreference.PRIMARY,
        "secondary": ReadPreference.SECONDARY,
    }[target]

    db = client.get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"], wtimeout=WTIMEOUT_MS),
        read_concern=ReadConcern(config["read_concern"]),
        read_preference=read_preference,
    )
    return db[collection_name]

#### Preparing Test Data

The helper functions below create and reset the student and course documents used in the experiments.

`SETUP_WRITE_CONCERN` is configured with `w=3`, meaning MongoDB waits for acknowledgement from all three replica-set members before confirming the reset operation. This ensures that every node, including `mongo3`, begins each experiment with the same baseline data before replication is paused.

The `reset_document()` function creates a new document or replaces an existing document with the same identifier. It uses the strong setup write concern to ensure that the reset is replicated across all nodes.

The `make_student()` and `make_course()` functions generate standard test documents with default values. Additional student fields can be changed using keyword overrides when required.

Each consistency configuration uses unique document identifiers, such as `S001-majority` or `C001-w1-local`. This prevents data and results from different configurations from being mixed.

In [ ]:
# Baseline writes wait for ALL three nodes, so every node starts from the same state.
# These must run BEFORE a partition is created: w=3 cannot be satisfied once a node is isolated.
SETUP_WRITE_CONCERN = WriteConcern(w=3, wtimeout=30000)


def reset_document(collection_name, id_field, document):
    """Create or overwrite a test document and wait until every node has it."""
    db = client.get_database(DB_NAME, write_concern=SETUP_WRITE_CONCERN)
    db[collection_name].replace_one({id_field: document[id_field]}, document, upsert=True)


def make_student(student_id, name, **overrides):
    student = {
        "student_id": student_id,
        "student_name": name,
        "student_status": "Undergraduate",
        "seniority": "Year 1",
        "academic_load": "Full Time",
    }
    student.update(overrides)
    return student


def make_course(course_id, name):
    return {
        "course_id": course_id,
        "course_name": name,
        "course_status": "ACTIVE",
        "vacancy": 30,
    }


def count_rollback_lines(node):
    """Count ROLLBACK entries in a node's Docker log."""
    logs = subprocess.run(["docker", "logs", node], capture_output=True, text=True)
    return (logs.stdout + logs.stderr).count("ROLLBACK")

### 2.2 Shared Reading and Consistency Reporting Functions

In [5]:
def read_field(collection, query, field, session=None):
    """
    Read one field of one document.
    Returns TIMEOUT if the node could not serve the read within READ_TIMEOUT_MS
    (this is how a causal session blocks a read on a node that is behind).
    """
    try:
        doc = collection.find_one(query, session=session, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return TIMEOUT
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def read_field_repeatedly(collection, query, field, session=None, n=NUM_READS):
    return [read_field(collection, query, field, session) for _ in range(n)]


def read_node(node, query, field, collection_name="students"):
    """Read one field from a single node's own copy of the data."""
    try:
        doc = direct[node][DB_NAME][collection_name].find_one(query, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return TIMEOUT
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def try_write(collection, query, update, session=None):
    """Run an update and report whether the write concern was satisfied."""
    try:
        collection.update_one(query, update, session=session)
        return "acknowledged"
    except WTimeoutError:
        return f"NOT confirmed within {WTIMEOUT_MS} ms"
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"


def show(label, value):
    print(f"  {label:<34} {value}")


def show_reads(label, reads):
    """Print the reads as a table, one row per read."""
    table = pd.DataFrame({"read": range(1, len(reads) + 1), "value": reads})
    print(f"  {label}")
    print(table.to_string(index=False))


def header(title, config_name):
    config = CONFIGS[config_name]
    print(f"=== {title} | w={config['write_concern']}, "
          f"readConcern={config['read_concern']} ===")


def report_stale_reads(reads, expected):
    """Read-your-writes check: any value other than what we wrote is stale."""
    stale = sum(1 for r in reads if r not in (expected, TIMEOUT))
    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if stale else "no violation in this run"
    print(f"  RESULTS: {verdict}: {stale} stale, {timeouts} timed out, out of {len(reads)} reads")


def report_regressions(reads, order):
    """Monotonic-reads check: count reads that went back to an older value."""
    rank = {value: i for i, value in enumerate(order)}
    newest_seen = -1
    regressions = 0
    for value in reads:
        if value not in rank:  # TIMEOUT or error
            continue
        if rank[value] < newest_seen:
            regressions += 1
        newest_seen = max(newest_seen, rank[value])

    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if regressions else "no violation in this run"
    print(f"  RESULTS: {verdict}: {regressions} went backwards, {timeouts} timed out, "
          f"out of {len(reads)} reads")

### 2.3 Creating the partition

## Simulating a network Partition

**Goal:** isolate `mongo3` from its peers at the network layer, while keeping it reachable by our client — so we can observe what a client sees when it talks to a node that is isolated from the network but still running.

Using `docker network disconnect` removes the container's only network interface entirely, including the published port our client connects through, making the node completely unreachable. Instead, we apply `iptables` rules that drop only the traffic between `mongo3` and its two peers, leaving everything else untouched.

Since the MongoDB image doesn't include `iptables` and we don't want to modify the container itself, we apply these rules from a temporary helper container that borrows `mongo3`'s network namespace (`--network container:mongo3`). The rules take effect on `mongo3` and persist after the helper exits. Our client connects via the host-published port (`127.0.0.4:27017`), which reaches `mongo3` through Docker's port-forwarding (from the bridge gateway) — not from a peer IP. Since our rules only block the two peer IPs, client traffic is never touched.

In [6]:
NETADMIN_IMAGE = "dsa5208-netadmin"

# electionTimeoutMillis defaults to 10 s, so allow a margin either way.
PARTITION_TIMEOUT = 40      # how long to wait for the split to be detected
HEAL_TIMEOUT = 90           # how long to wait for a healed set to become healthy again


def build_netadmin_image():
    """
    Build the tiny helper image that carries an iptables binary.
    Alpine + iptables is about 12 MB and only builds once; afterwards Docker serves it
    from cache, so this is cheap to call at the top of every run.
    """
    dockerfile = "FROM alpine\nRUN apk add --no-cache iptables\n"
    subprocess.run(["docker", "build", "-q", "-t", NETADMIN_IMAGE, "-"],
                   input=dockerfile, text=True, check=True, capture_output=True)
    print(f"  helper image ready: {NETADMIN_IMAGE}")


def node_ip(node):
    """The container's address on the compose network, e.g. "172.18.0.2"."""
    result = subprocess.run(
        ["docker", "inspect", "-f",
         "{{range .NetworkSettings.Networks}}{{.IPAddress}} {{end}}", node],
        capture_output=True, text=True, check=True)
    return result.stdout.split()[0]


def netadmin(node, script):
    """
    Run a shell script inside `node`'s network namespace with NET_ADMIN.
    The helper container has its own filesystem but `node`'s interfaces, so any
    iptables rule it sets applies to `node` and stays after the helper exits.
    """
    result = subprocess.run(
        ["docker", "run", "--rm", "--network", f"container:{node}",
         "--cap-add", "NET_ADMIN", NETADMIN_IMAGE, "sh", "-c", script],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"netadmin on {node} failed: {result.stderr.strip()}")
    return result.stdout

### 2.4 Applying, Verifying and Restoring a Partition

`partition(minority)` takes the list of nodes to isolate. Rules go on the minority side in both
directions, that's enough to cut the link: a heartbeat from `mongo1` gets dropped as it arrives
at `mongo3` (`INPUT`), and `mongo3`'s reply gets dropped on the way out (`OUTPUT`). Neither side
sees the other after that.

A split only really counts once both sides have noticed it, so `partition` waits until the
majority side actually reports the minority as unreachable before returning, instead of just
sleeping for a fixed amount of time. `heal()` flushes the rules on every node, so it's safe to
call any time, including after a run that got interrupted partway through.

In [7]:
def partition(minority, wait=True):
    """
    Cut `minority` off from every other node at the IP layer, and wait until the
    split has been detected. Clients keep full access to every node.

    wait=False returns as soon as the rules are in place, before the nodes have
    noticed. Section 5 needs that window: an isolated primary only steps down
    after the election timeout, and the writes it accepts in the meantime are
    exactly the ones that have to be rolled back.
    """
    majority = [node for node in NODES if node not in minority]
    for node in minority:
        rules = "".join(
            f"iptables -I INPUT -s {node_ip(peer)} -j DROP; "
            f"iptables -I OUTPUT -d {node_ip(peer)} -j DROP; "
            for peer in majority
        )
        netadmin(node, rules)
    print(f"  [partition] {'+'.join(minority)}  ||  {'+'.join(majority)}")
    if wait:
        wait_until_partitioned(minority, majority)
    return majority


def heal():
    """Remove all partition rules from every node. Safe to run at any time."""
    for node in NODES:
        try:
            netadmin(node, "iptables -F INPUT; iptables -F OUTPUT")
        except RuntimeError:
            pass  # node not running: nothing to flush
    print("  [heal] partition rules flushed on all nodes")


def member_health(observer):
    """How `observer` sees every member of the set: {name: health}, health 0 = unreachable."""
    status = direct[observer].admin.command("replSetGetStatus")
    return {m["name"].split(":")[0]: m["health"] for m in status["members"]}


def wait_until_partitioned(minority, majority, timeout=PARTITION_TIMEOUT):
    """
    Wait until BOTH sides report the other as unreachable.

    Checking only one side is not enough: detection is not simultaneous, and a run that
    started while the minority still believed the set was whole would be measuring a
    half-formed split.
    """
    start = time.time()
    deadline = start + timeout
    while time.time() < deadline:
        try:
            majority_view = member_health(majority[0])
            minority_view = member_health(minority[0])
            if (all(majority_view.get(node) == 0 for node in minority)
                    and all(minority_view.get(node) == 0 for node in majority)):
                print(f"  [partition] both sides detected the split after "
                      f"{time.time() - start:.0f}s")
                return
        except PyMongoError:
            pass
        time.sleep(2)
    raise RuntimeError(f"partition of {minority} was not detected within {timeout}s")


def current_primary(timeout=60):
    """Wait until the replica set has a primary and return its name, e.g. "mongo1"."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return client.admin.command("hello")["primary"].split(":")[0]
        except (PyMongoError, KeyError):
            time.sleep(1)
    raise RuntimeError("No primary was elected in time")


def wait_until_caught_up(node, query, field, expected, collection_name="students",
                        timeout=HEAL_TIMEOUT):
    """
    After healing, poll a node's own copy until it converges on `expected`.

    A fixed sleep is not enough here: heartbeats have to resume and the node has to refetch
    the oplog it missed, which takes longer than the write itself did. Polling also measures
    how long convergence actually took.
    """
    start = time.time()
    deadline = start + timeout
    value = None
    while time.time() < deadline:
        value = read_node(node, query, field, collection_name)
        if value == expected:
            return value, time.time() - start
        time.sleep(1)
    return value, None


def show_caught_up(node, query, field, expected, collection_name="students"):
    """Report a healed node's convergence in one line."""
    value, seconds = wait_until_caught_up(node, query, field, expected, collection_name)
    if seconds is None:
        show(f"{node} after healing:", f"{value}  (did NOT converge within {HEAL_TIMEOUT}s)")
    else:
        show(f"{node} after healing:", f"{value}  (converged in {seconds:.0f}s)")


def wait_until_healthy(timeout=HEAL_TIMEOUT):
    """Wait until all three nodes see each other and exactly one primary exists."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            status = client.admin.command("replSetGetStatus")
            states = [m["stateStr"] for m in status["members"]]
            healthy = all(m["health"] == 1 for m in status["members"])
            if healthy and states.count("PRIMARY") == 1 and states.count("SECONDARY") == 2:
                return True
        except PyMongoError:
            pass
        time.sleep(2)
    return False

### 2.5 Cluster Status and Restoration

In [8]:
def show_members():
    """Print each member's state, health and how far its oplog has been applied."""
    try:
        members = client.admin.command("replSetGetStatus")["members"]
    except PyMongoError as error:
        print(f"  cannot reach the set through the replica-set client: {type(error).__name__}")
        return
    for m in members:
        reachable = "reachable" if m["health"] == 1 else "UNREACHABLE"
        print(f"  {m['name']:<16} {m['stateStr']:<12} {reachable:<12} {m.get('optimeDate')}")


def show_both_views(minority):
    """Print the set as each side of the split sees it: neither side has the whole picture."""
    majority = [node for node in NODES if node not in minority]
    for observer in (majority[0], minority[0]):
        try:
            health = member_health(observer)
            seen = ", ".join(f"{n}={'up' if h == 1 else 'DOWN'}" for n, h in health.items())
            print(f"  as seen by {observer}: {seen}")
        except PyMongoError as error:
            print(f"  as seen by {observer}: ERROR ({type(error).__name__})")


def ensure_minority_is_secondary():
    """
    The experiments isolate MINORITY_NODE, which only tests a minority split if that node
    is a secondary. If it is currently primary, step it down and wait for a new primary.
    """
    if current_primary() == MINORITY_NODE:
        print(f"  {MINORITY_NODE} is primary; stepping it down")
        try:
            direct[MINORITY_NODE].admin.command("replSetStepDown", 60)
        except PyMongoError:
            pass  # the connection is dropped by the step-down itself
        time.sleep(2)
    print("  Primary:", current_primary())


def restore_cluster():
    """
    Undo anything this notebook changes, in case a run was interrupted:
    start every container, flush all partition rules, wait for a healthy set,
    and make sure MINORITY_NODE is a secondary again.
    """
    for node in NODES:
        subprocess.run(["docker", "unpause", node], capture_output=True)
        subprocess.run(["docker", "start", node], capture_output=True)
    heal()
    ok = wait_until_healthy()
    print("  Replica set healthy:", "yes" if ok else "NO - check the cells above")
    ensure_minority_is_secondary()
    show_members()

### 2.6 Checking the Cluster

Run this first. It builds the helper image, clears any partition left over from an earlier run,
and confirms the set is a healthy 3-node replica set with `mongo3` as a secondary.

In [9]:
build_netadmin_image()
restore_cluster()

  helper image ready: dsa5208-netadmin
  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:24:26
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:24:26
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:24:26


## Experiment 1: Read-Your-Writes Consistency under Partition

The client writes a new `student_status` through the primary, then reads the same document back
from the secondaries. `mongo3` is in the minority, so it can never receive the write, but the
client can still be routed to it.

### Prediction

**Read-your-writes** requires that a client which has written a value never subsequently reads an
older one.

**Note:** Session refers to a causally consistent session.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | **May be violated.** The write reaches `mongo1`+`mongo2` and is acknowledged, but reading from `mongo3` just returns whatever the old value of `mongo3` before the split | **Expected to be upheld.** The causal consistency session remembers the write's timestamp and requires any later read to be at least that recent. Since `mongo3` can't catch up while partitioned, the read blocks and returns a TIMEOUT instead of returning stale data. |
| `w1-local` | **May be violated.** `w=1` acknowledges the write right away, and `local` reads just return data stored locally with no freshness check, hence `mongo3` hands back the old value immediately. | **Expected to be upheld.** Even with `local` reads, the session's timestamp requirement still applies, so we get a timeout when trying to read from `mongo3` stale data, not a wrong answer. |

In [10]:
def run_read_your_writes(config_name):
    header("Read-your-writes under partition", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S001-{config_name}"
    query = {"student_id": student_id}

    # Baseline must be on all three nodes before the split.
    reset_document("students", "student_id", make_student(student_id, "Robert Tan"))
    show("Baseline on all nodes:", read_node(MINORITY_NODE, query, "student_status"))

    partition([MINORITY_NODE])
    show_both_views([MINORITY_NODE])
    try:
        # Without a session
        update = {"$set": {"student_status": "Graduate"}}
        show("Write result (no session):", try_write(students_secondary, query, update))
        show(f"{MINORITY_NODE} own copy:", read_node(MINORITY_NODE, query, "student_status"))
        reads = read_field_repeatedly(students_secondary, query, "student_status")
        show_reads("Reads (no session):", reads)
        report_stale_reads(reads, expected="Graduate")

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            update = {"$set": {"student_status": "Post Doc"}}
            show("Write result (session):",
                 try_write(students_secondary, query, update, session=session))
            show("Session operationTime:", session.operation_time)
            reads = read_field_repeatedly(
                students_secondary, query, "student_status", session=session
            )
            show_reads("Reads (causal consistency session):", reads)
            report_stale_reads(reads, expected="Post Doc")
    finally:
        heal()

    show_caught_up(MINORITY_NODE, query, "student_status", "Post Doc")

In [11]:
run_read_your_writes("majority")

=== Read-your-writes under partition | w=majority, readConcern=majority ===
  Baseline on all nodes:             Undergraduate
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Write result (no session):         acknowledged
  mongo3 own copy:                   Undergraduate
  Reads (no session):
 read         value
    1      Graduate
    2      Graduate
    3 Undergraduate
    4      Graduate
    5      Graduate
    6      Graduate
    7      Graduate
    8 Undergraduate
    9      Graduate
   10 Undergraduate
  RESULTS: VIOLATION: 3 stale, 0 timed out, out of 10 reads
  Write result (session):            acknowledged
  Session operationTime:             Timestamp(1789223087, 2)
  Reads (causal consistency session):
 read    value
    1 Post Doc
    2 Post Doc
    3  TIMEOUT
    4 Post Doc
    5  TIMEOUT
    6 Post Doc
    7 Pos

In [12]:
run_read_your_writes("w1-local")

=== Read-your-writes under partition | w=1, readConcern=local ===
  Baseline on all nodes:             Undergraduate
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Write result (no session):         acknowledged
  mongo3 own copy:                   Undergraduate
  Reads (no session):
 read         value
    1      Graduate
    2      Graduate
    3 Undergraduate
    4      Graduate
    5      Graduate
    6      Graduate
    7 Undergraduate
    8 Undergraduate
    9 Undergraduate
   10 Undergraduate
  RESULTS: VIOLATION: 5 stale, 0 timed out, out of 10 reads
  Write result (session):            acknowledged
  Session operationTime:             Timestamp(1789223110, 2)
  Reads (causal consistency session):
 read    value
    1 Post Doc
    2  TIMEOUT
    3  TIMEOUT
    4  TIMEOUT
    5  TIMEOUT
    6 Post Doc
    7 Post Doc
    

### Interpretation

The run confirmed the prediction. With the secondary `mongo3` isolated, both configurations violated read-your-writes without a session: the client had just received an acknowledgement for its write, yet three of ten reads under `majority` and five under `w1-local` returned the pre-partition value `Undergraduate`. The violation under `writeConcern: "majority"` shows that the cause was read routing, not durability. The write was safely committed on the majority side, but `readConcern: "majority"` on `mongo3` means majority-committed as far as `mongo3` knows, and its knowledge stopped at the moment of the split. Compared with the other notebooks, this is the only scenario in which a node was both reachable and wrong: in the normal-operation notebook, the stalled node served stale reads while the set remained fully connected, and in the failover notebook, the paused node could not serve reads at all.

With the causal session, read-your-writes held in both configurations. Every read either returned the new value or timed out, because `mongo3` could not show that it had reached the session's cluster time and refused to answer rather than answer incorrectly. Three of ten session reads timed out under `majority` and five under `w1-local`, roughly in line with the driver selecting between the two secondaries, only one of which was isolated. Because the primary remained on the majority side and no write was rolled back, `w1-local` behaved the same as `majority` in this scenario. After healing, `mongo3` caught up through the oplog without a rollback, within 1 second and 3 seconds respectively, confirming that isolating a secondary caused temporary staleness but no divergent history.

In [13]:
# Cleanup: reverse the partition and confirm a healthy 3-node set before the next experiment
restore_cluster()

  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:26:06
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:26:06
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:25:56


## Experiment 2. Monotonic-Reads Consistency under Partition

`seniority` ranking is advanced from `Year 1` to `Year 5` on the majority side while `mongo3` is isolated, then
the client reads the same document ten times from the secondaries. Successive reads alternate
between a node that has all five writes and a node frozen at `Year 1`.

**What to look for:** A monotonic-read violation occurs when a later read returns an older seniority value than a value already observed, for example:

```text
Year 1 → Year 5 → Year 1
```

### Prediction

**Monotonic reads** requires that once a client has seen a value, later reads never show an
older one.

**Note:** Session refers to a causally consistent session.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | **May be violated.** Reads land alternately on `mongo2` (`Year 5`) and `mongo3` (`Year 1`), so the sequence goes backwards. Each individual read is fine on its own but the sequence across nodes shows regression from `Year 5` to `Year 1`, since nothing ties one read to the next. | **Expected to be upheld.** Once a read has come back with `Year 5`, the session requires any later read to be at least that recent. Since `mongo3` can't catch up while partitioned, a read routed there blocks and returns TIMEOUT instead of going backwards. |
| `w1-local` | **May be violated.** `local` reads on `mongo2` also pick up uncommitted intermediate values, so the sequence may fluctuate and drop back to `Year 1` repeatedly. | **Expected to be upheld.** The session's timestamp requirement still applies, so we get a timeout when trying to read stale data from `mongo3`, not a step backwards. |

In [14]:
SENIORITY_ORDER = ["Year 1", "Year 2", "Year 3", "Year 4", "Year 5"]


def run_monotonic_reads(config_name):
    header("Monotonic reads under partition", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S002-{config_name}"
    query = {"student_id": student_id}

    reset_document("students", "student_id", make_student(student_id, "Feyd Johnston"))
    show("Baseline on all nodes:", read_node(MINORITY_NODE, query, "seniority"))

    partition([MINORITY_NODE])
    show_both_views([MINORITY_NODE])
    try:
        for year in SENIORITY_ORDER[1:]:
            students_secondary.update_one(query, {"$set": {"seniority": year}})
        show("Wrote seniority:", " -> ".join(SENIORITY_ORDER))
        show(f"{MINORITY_NODE} own copy:", read_node(MINORITY_NODE, query, "seniority"))

        # Without a session
        reads = read_field_repeatedly(students_secondary, query, "seniority")
        show_reads("Reads (no session):", reads)
        report_regressions(reads, SENIORITY_ORDER)

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            reads = read_field_repeatedly(
                students_secondary, query, "seniority", session=session
            )
        show_reads("Reads (causal consistency session):", reads)
        report_regressions(reads, SENIORITY_ORDER)
    finally:
        heal()

    show_caught_up(MINORITY_NODE, query, "seniority", "Year 5")

In [15]:
run_monotonic_reads("majority")

=== Monotonic reads under partition | w=majority, readConcern=majority ===
  Baseline on all nodes:             Year 1
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Wrote seniority:                   Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
  mongo3 own copy:                   Year 1
  Reads (no session):
 read  value
    1 Year 1
    2 Year 5
    3 Year 1
    4 Year 1
    5 Year 5
    6 Year 1
    7 Year 5
    8 Year 5
    9 Year 5
   10 Year 1
  RESULTS: VIOLATION: 4 went backwards, 0 timed out, out of 10 reads
  Reads (causal consistency session):
 read   value
    1  Year 5
    2 TIMEOUT
    3 TIMEOUT
    4  Year 5
    5 TIMEOUT
    6 TIMEOUT
    7 TIMEOUT
    8 TIMEOUT
    9  Year 5
   10  Year 5
  RESULTS: no violation in this run: 0 went backwards, 6 timed out, out of 10 reads
  [heal] partition rules flushed on a

In [16]:
run_monotonic_reads("w1-local")

=== Monotonic reads under partition | w=1, readConcern=local ===
  Baseline on all nodes:             Year 1
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 10s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Wrote seniority:                   Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
  mongo3 own copy:                   Year 1
  Reads (no session):
 read  value
    1 Year 5
    2 Year 5
    3 Year 1
    4 Year 1
    5 Year 1
    6 Year 5
    7 Year 5
    8 Year 1
    9 Year 5
   10 Year 1
  RESULTS: VIOLATION: 5 went backwards, 0 timed out, out of 10 reads
  Reads (causal consistency session):
 read   value
    1  Year 5
    2  Year 5
    3 TIMEOUT
    4  Year 5
    5 TIMEOUT
    6  Year 5
    7 TIMEOUT
    8 TIMEOUT
    9 TIMEOUT
   10  Year 5
  RESULTS: no violation in this run: 0 went backwards, 5 timed out, out of 10 reads
  [heal] partition rules flushed on all nodes
 

### Interpretation

The run confirmed the prediction. With the secondary `mongo3` isolated, both configurations violated monotonic reads without a session. Because `mongo3` could not receive any writes during the partition, its value was permanently older, not merely momentarily behind. The regressions under `readConcern: "majority"` are not a bug: majority read concern guarantees that the returned data is durable, not that it is recent, and it does not control which secondary serves the read.

With the causal session, monotonic reads held in both configurations. The values returned never decreased, because each read had to satisfy the session's `afterClusterTime`, and reads that could not do so timed out instead of returning `Year 1`. This is consistent with those reads being routed to the isolated `mongo3`. The guarantee came at a clear price: the session refused to answer, which shows that under partition, consistency is preserved at the cost of availability. Because only a secondary was isolated, the primary stayed on the majority side and no write was rolled back, which is why `w1-local` behaved the same as `majority` in this scenario.

In [17]:
# Cleanup: reverse the partition and confirm a healthy 3-node set before the next experiment
restore_cluster()

  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:27:26
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:27:26
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:27:16


## 3. Monotonic-Writes Consistency under Partition

Two qualifications are appended sequentially to the student's `academic_history` while replication on `mongo3` is paused. The academic history is then compared across the primary, the stalled replica, and `mongo3` after replication resumes.

**What to look for:** Whether `mongo3` applies the qualifications in the same order as the primary after it catches up. An incomplete history while `mongo3` is isolated indicates replication delay; a reversed qualification order would indicate a monotonic-writes violation.

### Prediction

**Monotonic writes** requires that writes issued by one client are applied in the order the
client issued them, on every node.

| Configuration | Prediction |
|---|---|
| `"majority"` | **Upheld** Both writes go to the same primary, so they land in one ordered oplog. `mongo3` isn't just behind on both writes, it's stopped at an earlier point in that same sequence. Once the partition heals, it replays the oplog from where it left off, so every state it passes through is a prefix of the primary's history: `[B.Sc.]` → `[B.Sc., M.Sc.]` → `[B.Sc., M.Sc., PhD]`, never `[B.Sc., PhD]`. |
| `w=1-local`| **Upheld** Ordering comes from how the oplog is replayed, not from write concern. `w=1` just weakens how durable each write is and how fresh a read has to be, it doesn't change the order a secondary applies things in. The difference here should just be timing, not order. |

In [18]:
def run_monotonic_writes(config_name):
    header("Monotonic writes under partition", config_name)
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S003-{config_name}"
    query = {"student_id": student_id}

    reset_document("students", "student_id", make_student(
        student_id, "Alan White",
        student_status="Post Doc",
        academic_history=["B.Sc. Computer Science"],
    ))
    show("Baseline on all nodes:", read_node(MINORITY_NODE, query, "academic_history"))

    partition([MINORITY_NODE])
    show_both_views([MINORITY_NODE])
    try:
        for degree in ["M.Sc. Data Science", "PhD Information Systems"]:
            show(f"Write {degree!r}:",
                 try_write(students_primary, query, {"$push": {"academic_history": degree}}))

        show("Primary:", read_field(students_primary, query, "academic_history"))
        show(f"{MINORITY_NODE} (partitioned):",
             read_node(MINORITY_NODE, query, "academic_history"))

        reads = read_field_repeatedly(students_secondary, query, "academic_history")
        show_reads("Reads from secondaries (no session):", reads)
    finally:
        heal()

    # Sample the isolated node while it replays the oplog: every state it shows should be
    # a prefix of the primary's history, never a reordering.
    expected = ["B.Sc. Computer Science", "M.Sc. Data Science", "PhD Information Systems"]
    catch_up = []
    deadline = time.time() + 30
    while time.time() < deadline:
        value = read_node(MINORITY_NODE, query, "academic_history")
        if not catch_up or value != catch_up[-1]:
            catch_up.append(value)
        if value == expected:
            break
        time.sleep(0.5)

    show_reads(f"{MINORITY_NODE} states while catching up:", catch_up)
    out_of_order = [v for v in catch_up
                    if isinstance(v, list) and v != expected[:len(v)]]
    verdict = "VIOLATION" if out_of_order else "no violation in this run"
    print(f"  RESULTS: {verdict}: {len(out_of_order)} of {len(catch_up)} observed states "
          f"were not a prefix of the primary's history")

In [19]:
run_monotonic_writes("majority")

=== Monotonic writes under partition | w=majority, readConcern=majority ===
  Baseline on all nodes:             ['B.Sc. Computer Science']
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Write 'M.Sc. Data Science':        acknowledged
  Write 'PhD Information Systems':   acknowledged
  Primary:                           ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
  mongo3 (partitioned):              ['B.Sc. Computer Science']
  Reads from secondaries (no session):
 read                                                                 value
    1                                              [B.Sc. Computer Science]
    2                                              [B.Sc. Computer Science]
    3 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    4 [B.Sc. Computer Science, M.S

In [20]:
run_monotonic_writes("w1-local")

=== Monotonic writes under partition | w=1, readConcern=local ===
  Baseline on all nodes:             ['B.Sc. Computer Science']
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Write 'M.Sc. Data Science':        acknowledged
  Write 'PhD Information Systems':   acknowledged
  Primary:                           ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
  mongo3 (partitioned):              ['B.Sc. Computer Science']
  Reads from secondaries (no session):
 read                                                                 value
    1                                              [B.Sc. Computer Science]
    2 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    3                                              [B.Sc. Computer Science]
    4 [B.Sc. Computer Science, M.Sc. Data Sc

### Interpretation

The run confirmed the prediction. With the secondary `mongo3` isolated, every state sampled from `mongo3` during catch-up was a prefix of `["B.Sc.", "M.Sc.", "PhD"]`: it moved directly from the baseline to the full array in the correct order. The intermediate state was not captured, most likely because both writes were applied within a single sampling interval. The stale baseline values returned during the partition therefore reflected `mongo3`'s lag, not incorrectly ordered writes. This follows from the single-primary oplog. The primary assigns each write one position in a total order, and a secondary applies entries strictly in that order, so the partition could delay `mongo3` but could not change the order in which it applied the writes.

The `w1-local` and `majority` configurations behaved identically in terms of ordering. They differed only in which reads happened to return the baseline, which reflects recency rather than ordering. This shows that ordering guarantees are separate from the recency and durability guarantees controlled by the write and read concerns. The result holds because only the majority side accepted writes while the secondary was isolated. Experiment 5 removes that assumption by isolating the primary instead.

In [21]:
# Cleanup: reverse the partition and confirm a healthy 3-node set before the next experiment
restore_cluster()

  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:27:58
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:27:58
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:27:58


## Experiment 4. Writes-Follow-Reads Consistency under Partition

The course's `course_status` is changed to `CANCELLED` on the majority side while `mongo3` is partitioned and still holds `ACTIVE`. The client then reads `course_status`, possibly from the stale `mongo3`, and writes a student `enrollment_note` to the primary containing the value returned by that read. The experiment compares this read-then-write sequence with and without a causally consistent session. In the session case, the session first observes the cancellation on the primary, so that every later operation is ordered after it.

**What to look for:** Whether the enrollment note is based on the current course status or on the superseded `ACTIVE` value served by the partitioned `mongo3`, and whether a causally consistent session prevents reads that are older than the state the session has already observed. The isolated `mongo3` may return `TIMEOUT` rather than satisfy the causal dependency, and once the partition heals, it should hold both the cancellation and the note that depends on it.

### Prediction

**Writes-follow-reads** (session causality) requires that a write which was made on the basis of
a read is ordered after the state that read observed, everywhere it becomes visible.

**Note:** Session refers to a causally consistent session.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | **May be violated.** If the read is routed to `mongo3` it returns `ACTIVE`, a state that's already been superseded by `CANCELLED` on the majority side. The note the client then writes records a course state that no longer exists, and it's committed after the cancellation in the oplog. The read-then-write chain is inverted relative to what actually happened: the write follows a read of the past. | **Expected to be upheld.** The session's timestamp only reflects what it's already observed, so a brand new session has no timestamp yet, and its very first read can still be served stale by `mongo3`. Once the session has seen the cancellation though, every later read routed to `mongo3` must be at least that recent, can't be, and returns TIMEOUT. Reading the note back afterward is likewise guaranteed to include the note's own timestamp. So the experiment needs to establish the session's context first, then test the chain. |
| `w1-local` | **May be violated.** `local` gives the read no freshness requirement at all, so the stale `ACTIVE` comes back immediately and the inverted chain gets created without any delay. | **Expected to be upheld.** with the same first-operation caveat as above. The timestamp requirement is enforced regardless of read concern, so once it's set, the stale read times out rather than feeding into the write. |

In [22]:
def run_writes_follow_reads(config_name):
    header("Writes-follow-reads under partition", config_name)
    courses_secondary = get_collection("courses", config_name, "secondary")
    courses_primary = get_collection("courses", config_name, "primary")
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")

    course_id = f"C001-{config_name}"
    course_name = "Programming Methodology"
    student_id = f"S004-{config_name}"
    course_query = {"course_id": course_id}
    student_query = {"student_id": student_id}

    reset_document("courses", "course_id", make_course(course_id, course_name))
    reset_document("students", "student_id",
                   make_student(student_id, "Feyd Johnston", enrollment_note=None))
    show("Baseline course_status:", read_node(MINORITY_NODE, course_query,
                                              "course_status", "courses"))

    partition([MINORITY_NODE])
    show_both_views([MINORITY_NODE])
    try:
        # The majority side cancels the course; the minority node cannot learn about it.
        show("Cancel the course:",
             try_write(courses_primary, course_query,
                       {"$set": {"course_status": "CANCELLED"}}))
        show("Primary course_status:", read_field(courses_primary, course_query, "course_status"))
        show(f"{MINORITY_NODE} course_status:",
             read_node(MINORITY_NODE, course_query, "course_status", "courses"))

        # Which node answers a secondary read is not under our control, so record the
        # spread first: this is the disagreement the client is exposed to.
        statuses = read_field_repeatedly(courses_secondary, course_query, "course_status")
        show_reads("Course reads (no session):", statuses)
        print(f"  {statuses.count('ACTIVE')} of {len(statuses)} reads still said ACTIVE")

        # Without a session: take the read a client routed to the isolated node would get,
        # and write a note based on it. Using the minority node explicitly keeps the
        # experiment deterministic instead of depending on which secondary was picked.
        status = read_node(MINORITY_NODE, course_query, "course_status", "courses")
        show("Enrolling on the basis of:", f"{status} (read from {MINORITY_NODE})")
        students_primary.update_one(student_query, {"$set": {
            "enrollment_note": f"Enrolled in {course_name} while course_status was: {status}"
        }})
        show("Note now on the primary:",
             read_field(students_primary, student_query, "enrollment_note"))
        stale_chain = status != "CANCELLED"
        print(f"  RESULTS: {'VIOLATION' if stale_chain else 'no violation in this run'}: "
              f"the note was written on the basis of a "
              f"{'superseded' if stale_chain else 'current'} course state")

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            # A new session carries no cluster time, so its first read could legitimately be
            # served stale. Establish the causal context first: the session observes the
            # cancellation, and everything after this must be ordered after it.
            seen = read_field(courses_primary, course_query, "course_status", session=session)
            show("Session first observed:", seen)
            show("Session cluster time:", session.cluster_time["clusterTime"]
                 if session.cluster_time else None)

            statuses = read_field_repeatedly(
                courses_secondary, course_query, "course_status", session=session
            )
            show_reads("Course reads (causal consistency session):", statuses)
            stale = [s for s in statuses if s not in (TIMEOUT, "CANCELLED")]
            verdict = "VIOLATION" if stale else "no violation in this run"
            print(f"  RESULTS: {verdict}: {len(stale)} reads returned a superseded state, "
                  f"{statuses.count(TIMEOUT)} were refused with {TIMEOUT} rather than "
                  f"answered from before the split")

            # The write is now seeded by a read the session has already ordered.
            status = next((s for s in statuses if s != TIMEOUT), seen)
            show("Session enrolling on:", status)
            students_primary.update_one(student_query, {"$set": {
                "enrollment_note": f"Session enrollment, course_status: {status}"
            }}, session=session)
            notes = read_field_repeatedly(
                students_secondary, student_query, "enrollment_note", session=session
            )
            show_reads("Session read-back of the note:", notes)
            lost = [n for n in notes if n != TIMEOUT and "Session enrollment" not in str(n)]
            verdict = "VIOLATION" if lost else "no violation in this run"
            print(f"  RESULTS: {verdict}: {len(lost)} read-backs failed to show the note the "
                  f"session had just written, {notes.count(TIMEOUT)} were refused")
    finally:
        heal()

    show_caught_up(MINORITY_NODE, course_query, "course_status", "CANCELLED", "courses")
    show(f"{MINORITY_NODE} note after healing:",
         read_node(MINORITY_NODE, student_query, "enrollment_note"))

In [23]:
run_writes_follow_reads("majority")

=== Writes-follow-reads under partition | w=majority, readConcern=majority ===
  Baseline course_status:            ACTIVE
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Cancel the course:                 acknowledged
  Primary course_status:             CANCELLED
  mongo3 course_status:              ACTIVE
  Course reads (no session):
 read     value
    1 CANCELLED
    2 CANCELLED
    3 CANCELLED
    4 CANCELLED
    5 CANCELLED
    6 CANCELLED
    7 CANCELLED
    8    ACTIVE
    9    ACTIVE
   10    ACTIVE
  3 of 10 reads still said ACTIVE
  Enrolling on the basis of:         ACTIVE (read from mongo3)
  Note now on the primary:           Enrolled in Programming Methodology while course_status was: ACTIVE
  RESULTS: VIOLATION: the note was written on the basis of a superseded course state
  Session first observed:            C

In [24]:
run_writes_follow_reads("w1-local")

=== Writes-follow-reads under partition | w=1, readConcern=local ===
  Baseline course_status:            ACTIVE
  [partition] mongo3  ||  mongo1+mongo2
  [partition] both sides detected the split after 12s
  as seen by mongo1: mongo1=up, mongo2=up, mongo3=DOWN
  as seen by mongo3: mongo1=DOWN, mongo2=DOWN, mongo3=up
  Cancel the course:                 acknowledged
  Primary course_status:             CANCELLED
  mongo3 course_status:              ACTIVE
  Course reads (no session):
 read     value
    1    ACTIVE
    2    ACTIVE
    3 CANCELLED
    4    ACTIVE
    5 CANCELLED
    6 CANCELLED
    7    ACTIVE
    8    ACTIVE
    9    ACTIVE
   10 CANCELLED
  6 of 10 reads still said ACTIVE
  Enrolling on the basis of:         ACTIVE (read from mongo3)
  Note now on the primary:           Enrolled in Programming Methodology while course_status was: ACTIVE
  RESULTS: VIOLATION: the note was written on the basis of a superseded course state
  Session first observed:            CANCELLED
 

### Interpretation

The run confirmed the predicted behaviour, although the no-session outcome is better classified as a stale-read anomaly than as a Write-Follow-Reads violation. With the secondary `mongo3` isolated, both configurations allowed the client, when no session was used, to base its enrollment note on the superseded `ACTIVE` state that `mongo3` still held. However, the note was written to the primary after the cancellation, so it was still ordered after the state the client had read, and the ordering that Write-Follow-Reads requires was preserved.

With the causal session, ordering was preserved relative to the `CANCELLED` state the session had already observed. Reads that could not satisfy the session's `afterClusterTime` requirement, consistent with reads routed to the isolated `mongo3`, timed out instead of returning `ACTIVE`, so the note was based on the current status. Every read-back of the note either returned it or timed out, and after healing, `mongo3` held both the cancellation and the note. This protection applied only after the session had observed the newer state, since causal consistency orders a client's operations relative to its own history rather than guaranteeing freshness. Because only a secondary was isolated, the primary stayed on the majority side throughout and no write was rolled back, which is why the `w1-local` configuration behaved the same as `majority` in this scenario.

In [25]:
# Cleanup: reverse the partition and confirm a healthy 3-node set before the next experiment
restore_cluster()

  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:30:06
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:30:06
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:30:06


## Experiment 5. Majority-vs-Minority Split - Isolating the Primary

Experiments 1-4 isolate a secondary, so only one side of the split ever accepts writes. This section
isolates the current primary. This results in two groups, one containing the old primary and one containing a majority, each unable to tell
whether the other side has failed or is simply unreachable.

What MongoDB does about it:

- The majority side (two nodes) can hold an election and promote a new primary within roughly
  one election timeout, and keeps serving writes normally.
- The minority side (the old primary alone) can't reach a majority, so it steps itself down after
  the same timeout. This is due to the fact that a minority can never keep a writable
  primary, so the set never ends up with two primaries accepting conflicting writes for long.
- The dangerous window is the gap before the step-down. During it, the old primary still believes
  it's primary and will accept `w=1` writes. Those writes reach no other node, so once the
  partition heals, they get rolled back, written, acknowledged, and then silently removed.

The experiment below writes into that window on purpose, then heals the partition and checks the
old primary's log for a rollback.

### Prediction

| Configuration| Original Primary | New Primary | Prediction |
|---|---|---|---|
| `majority` | **Steps down** | **Elected** on majority side | **Upheld** The write is never acknowledged: no majority is reachable from the minority side, so it fails with a write-concern timeout (or a not-primary error if the step-down has already happened). The old primary still applies it to its own oplog, so a rollback is expected here too, but the client never received an acknowledgement, so nothing it was promised is lost. The difference from `w=1` is whether the client was misled about the write. |
| `w1-local` | **Steps down** | **Elected** on majority side | **Violated** The write into the pre-step-down window is acknowledged and then lost. `w=1` only needs the node the client is talking to, and that node is about to find out it's no longer primary. A rollback is expected in the old primary's log after healing, with the value gone. This is a monotonic-writes and a read-your-writes violation at the same time, caused purely by the partition. |

In [26]:
def direct_collection(node, config_name, collection_name="students"):
    """A collection handle bound to one specific node, with that config's concerns."""
    config = CONFIGS[config_name]
    db = direct[node].get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"], wtimeout=WTIMEOUT_MS),
        read_concern=ReadConcern(config["read_concern"]),
    )
    return db[collection_name]


def wait_until_stepped_down(node, timeout=60):
    """Wait until an isolated primary has demoted itself, and report how long it took."""
    start = time.time()
    deadline = start + timeout
    while time.time() < deadline:
        try:
            if not direct[node].admin.command("hello").get("isWritablePrimary"):
                return time.time() - start
        except PyMongoError:
            pass
        time.sleep(1)
    return None


def run_isolated_primary(config_name):
    header("Majority-vs-minority split (primary isolated)", config_name)

    old_primary = current_primary()
    majority_side = [node for node in NODES if node != old_primary]
    rollback_before = count_rollback_lines(old_primary)

    student_id = f"S005-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(
        student_id, "Alan White", academic_history=["B.Sc. Computer Science"]
    ))
    show("Old primary:", old_primary)
    show("Majority side:", "+".join(majority_side))

    # wait=False: write into the window before the old primary notices it is alone.
    partition([old_primary], wait=False)
    try:
        minority_students = direct_collection(old_primary, config_name)
        show("Write to the isolated primary:",
             try_write(minority_students, query,
                       {"$push": {"academic_history": "M.Sc. MINORITY SIDE"}}))
        show(f"{old_primary} now holds:", read_node(old_primary, query, "academic_history"))

        # The old primary must give up its role once it cannot see a majority.
        seconds = wait_until_stepped_down(old_primary)
        show("Old primary stepped down after:",
             f"{seconds:.0f}s" if seconds is not None else "did NOT step down")
        show("Write after the step-down:",
             try_write(minority_students, query,
                       {"$push": {"academic_history": "PhD MINORITY SIDE"}}))

        # Meanwhile the majority side elects a new primary and keeps working.
        wait_until_partitioned([old_primary], majority_side)
        new_primary = None
        deadline = time.time() + 60
        while time.time() < deadline and new_primary is None:
            for node in majority_side:
                try:
                    if direct[node].admin.command("hello").get("isWritablePrimary"):
                        new_primary = node
                except PyMongoError:
                    pass
            time.sleep(1)
        show("New primary on majority side:", new_primary or "none elected")

        if new_primary:
            majority_students = direct_collection(new_primary, config_name)
            show("Write to the new primary:",
                 try_write(majority_students, query,
                           {"$push": {"academic_history": "M.Sc. MAJORITY SIDE"}}))
            # A w="majority" write issued to a just-elected primary can exceed a short
            # wtimeout while still being applied and committed moments later, so check
            # what the majority side actually holds rather than trusting the verdict.
            for node in majority_side:
                show(f"{node} holds:", read_node(node, query, "academic_history"))
        show(f"{old_primary} holds:", read_node(old_primary, query, "academic_history"))
        show("Two different histories:",
             "yes - the split is visible to clients on both sides")
    finally:
        heal()

    # Healing forces the old primary to reconcile with the majority side's history.
    ok = wait_until_healthy()
    show("Replica set healthy again:", "yes" if ok else "NO")
    time.sleep(CATCHUP_SECONDS)
    show(f"{old_primary} after healing:", read_node(old_primary, query, "academic_history"))
    show("Current primary holds:", read_field(
        get_collection("students", config_name, "primary"), query, "academic_history"))
    rolled_back = count_rollback_lines(old_primary) > rollback_before
    show(f"{old_primary} rolled back:", "yes" if rolled_back else "no")

In [27]:
run_isolated_primary("w1-local")

=== Majority-vs-minority split (primary isolated) | w=1, readConcern=local ===
  Old primary:                       mongo1
  Majority side:                     mongo2+mongo3
  [partition] mongo1  ||  mongo2+mongo3
  Write to the isolated primary:     acknowledged
  mongo1 now holds:                  ['B.Sc. Computer Science', 'M.Sc. MINORITY SIDE']
  Old primary stepped down after:    10s
  Write after the step-down:         ERROR (NotPrimaryError)
  [partition] both sides detected the split after 2s
  New primary on majority side:      mongo3
  Write to the new primary:          acknowledged
  mongo2 holds:                      ['B.Sc. Computer Science']
  mongo3 holds:                      ['B.Sc. Computer Science', 'M.Sc. MAJORITY SIDE']
  mongo1 holds:                      ['B.Sc. Computer Science', 'M.Sc. MINORITY SIDE']
  Two different histories:           yes - the split is visible to clients on both sides
  [heal] partition rules flushed on all nodes
  Replica set healthy again

In [28]:
run_isolated_primary("majority")

=== Majority-vs-minority split (primary isolated) | w=majority, readConcern=majority ===
  Old primary:                       mongo3
  Majority side:                     mongo1+mongo2
  [partition] mongo3  ||  mongo1+mongo2
  Write to the isolated primary:     NOT confirmed within 3000 ms
  mongo3 now holds:                  ['B.Sc. Computer Science', 'M.Sc. MINORITY SIDE']
  Old primary stepped down after:    7s
  Write after the step-down:         ERROR (NotPrimaryError)
  [partition] both sides detected the split after 2s
  New primary on majority side:      mongo1
  Write to the new primary:          NOT confirmed within 3000 ms
  mongo1 holds:                      ['B.Sc. Computer Science', 'M.Sc. MAJORITY SIDE']
  mongo2 holds:                      ['B.Sc. Computer Science', 'M.Sc. MAJORITY SIDE']
  mongo3 holds:                      ['B.Sc. Computer Science', 'M.Sc. MINORITY SIDE']
  Two different histories:           yes - the split is visible to clients on both sides
  [heal] 

### Interpretation

The run confirmed the prediction. In both configurations, the isolated primary accepted a write before stepping down, and that minority-side write was rolled back once the partition healed. The difference lay in whether the client had been told that the write succeeded.

Under `w: 1`, the isolated primary acknowledged the write immediately, preserving availability, but the acknowledgement was later invalidated when the write was rolled back. This happened because `w: 1` acknowledges a write before it has been replicated, and MongoDB resolves a split by discarding the history that a majority never received. Under `"majority"`, the client was protected because the minority-side write was never acknowledged, so the client could not build on it. This protection came at the cost of availability.

The results also show that a timeout does not reveal the final outcome of a write. Under `"majority"`, one timed-out write was rolled back, while another survived and was committed on the majority side. A client therefore cannot assume that a timed-out write either succeeded or failed.

In [29]:
# Cleanup: reverse the partition and confirm a healthy 3-node set
restore_cluster()

  [heal] partition rules flushed on all nodes
  Replica set healthy: yes
  Primary: mongo1
  mongo1:27017     PRIMARY      reachable    2026-09-12 14:31:44
  mongo2:27017     SECONDARY    reachable    2026-09-12 14:31:44
  mongo3:27017     SECONDARY    reachable    2026-09-12 14:31:44


## Clean Up

Leaves the set in the same state the other two notebooks expect: three healthy nodes, no
partition rules, and `mongo3` a secondary.

In [ ]:
restore_cluster()
print("\niptables rules remaining on each node:")
for node in NODES:
    rules = netadmin(node, "iptables -S INPUT; iptables -S OUTPUT")
    drops = [line for line in rules.splitlines() if "DROP" in line]
    print(f"  {node}: {len(drops)} drop rules")